<a href="https://colab.research.google.com/github/NsElgezawy/DataSci-Studio/blob/main/datasci_notaa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [109]:
# !pip install pandas matplotlib seaborn missingno pyngrok findspark streamlit xgboost -q

### data loader file

In [110]:
%%writefile data_loader.py
import os
import streamlit as st

import findspark
findspark.init()
from pyspark.sql import SparkSession


@st.cache_resource
def get_spark_session():
    spark = SparkSession.builder \
        .appName("DataSciStudio") \
        .config("spark.sql.adaptive.enabled", "true") \
        .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
        .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
        .getOrCreate()
    return spark


spark = get_spark_session()


def load_csv_spark(file, nrows=None):
    df = spark.read.csv(file, header=True, inferSchema=True)
    if nrows and nrows > 0:
        df = df.limit(nrows)
    return df


def load_kaggle_spark(path: str, filename: str, nrows=None):
    full_path = os.path.join(path.strip(), filename.strip())
    if not os.path.exists(full_path):
        dataset_id = path.strip().rstrip('/')
        os.system(f"kaggle datasets download -d {dataset_id} --unzip -p /tmp/kaggle_data/")
        full_path = f"/tmp/kaggle_data/{filename.strip()}"
    df = spark.read.csv(full_path, header=True, inferSchema=True)
    if nrows and nrows > 0:
        df = df.limit(nrows)
    return df

Overwriting data_loader.py


### UI file

In [111]:
%%writefile ui.py
import streamlit as st
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
from datetime import datetime

# ==============================================================================
# THEME SYSTEM
# ==============================================================================
THEMES = {
    "dark": {
        "bg":        "#0a0a0f",
        "surface":   "#111118",
        "surface2":  "#1a1a24",
        "border":    "#2a2a3a",
        "text":      "#e8e8f0",
        "muted":     "#888899",
        "primary":   "#7c6af7",
        "secondary": "#f7c26a",
        "accent":    "#6af7c2",
        "danger":    "#db2121",
        "info":      "#6aa8f7",
        "highlight": "#f76af7",
        "success":   "#aaf76a",
    },
    "light": {
        "bg":        "#f8f9fc",
        "surface":   "#ffffff",
        "surface2":  "#f1f3f9",
        "border":    "#e0e3eb",
        "text":      "#1f2937",
        "muted":     "#6b7280",
        "primary":   "#5b4df5",
        "secondary": "#f4b740",
        "accent":    "#22c7a9",
        "danger":    "#cf2929",
        "info":      "#3b82f6",
        "highlight": "#d946ef",
        "success":   "#84cc16",
    },
}

PALETTES = {
    "dark":  ["#7c6af7", "#f7c26a", "#6af7c2", "#f76a6a", "#6aa8f7", "#f76af7", "#aaf76a"],
    "light": ["#5b4df5", "#f4b740", "#22c7a9", "#e54848", "#3b82f6", "#d946ef", "#84cc16"],
}


def get_theme():
    mode = st.session_state.get("theme_mode", "dark")
    return THEMES[mode], PALETTES[mode], mode


def inject_css(t, mode):
    """Inject all CSS using the active theme dict `t`."""
    scrollbar_thumb = t["primary"]
    scrollbar_track = t["surface2"]
    input_shadow    = f"0 0 0 2px {t['primary']}40"

    st.markdown(f"""
<style>
@import url('https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&family=DM+Sans:wght@300;400;500;700&display=swap');

:root {{
    --bg:        {t["bg"]};
    --surface:   {t["surface"]};
    --surface2:  {t["surface2"]};
    --border:    {t["border"]};
    --text:      {t["text"]};
    --muted:     {t["muted"]};
    --primary:   {t["primary"]};
    --secondary: {t["secondary"]};
    --accent:    {t["accent"]};
    --danger:    {t["danger"]};
    --info:      {t["info"]};
    --highlight: {t["highlight"]};
    --success:   {t["success"]};
}}

/* ── Global ── */
html, body, [class*="css"] {{
    font-family: 'DM Sans', sans-serif;
    background-color: var(--bg) !important;
    color: var(--text) !important;
}}
.stApp {{ background: var(--bg) !important; }}
.main .block-container {{ padding-top: 1.5rem; }}

/* ── Sidebar ── */
section[data-testid="stSidebar"] {{
    background: var(--surface) !important;
    border-right: 1px solid var(--border) !important;
}}
section[data-testid="stSidebar"] * {{ color: var(--text) !important; }}
section[data-testid="stSidebar"] .stMarkdown h1,
section[data-testid="stSidebar"] .stMarkdown h2,
section[data-testid="stSidebar"] .stMarkdown h3 {{
    color: var(--primary) !important;
    font-family: 'Space Mono', monospace;
    font-size: 0.72rem;
    letter-spacing: 0.15em;
    text-transform: uppercase;
}}

/* ── Tabs ── */
.stTabs [data-baseweb="tab-list"] {{
    background: var(--surface2);
    border-radius: 10px;
    padding: 4px;
    gap: 4px;
    border: 1px solid var(--border);
}}
.stTabs [data-baseweb="tab"] {{
    background: transparent;
    color: var(--muted);
    border-radius: 7px;
    font-family: 'Space Mono', monospace;
    font-size: 0.78rem;
    letter-spacing: 0.04em;
    padding: 8px 20px;
    border: none;
    transition: color 0.15s;
}}
.stTabs [aria-selected="true"] {{
    background: var(--primary) !important;
    color: #ffffff !important;
}}

/* ── Metrics ── */
div[data-testid="metric-container"] {{
    background: var(--surface2);
    border: 1px solid var(--border);
    border-radius: 12px;
    padding: 16px;
    transition: border-color 0.2s;
}}
div[data-testid="metric-container"]:hover {{
    border-color: var(--primary);
}}
div[data-testid="metric-container"] label {{
    color: var(--muted) !important;
    font-size: 0.72rem;
    font-family: 'Space Mono', monospace;
    letter-spacing: 0.05em;
    text-transform: uppercase;
}}
div[data-testid="metric-container"] div[data-testid="stMetricValue"] {{
    color: var(--secondary) !important;
    font-family: 'Space Mono', monospace;
    font-size: 1.4rem !important;
    font-weight: 700;
}}
div[data-testid="stMetricDelta"] {{ color: var(--accent) !important; }}

/* ── Buttons ── */
.stButton > button {{
    background: var(--primary) !important;
    color: #ffffff !important;
    border: none !important;
    border-radius: 8px !important;
    font-family: 'Space Mono', monospace !important;
    font-size: 0.76rem !important;
    letter-spacing: 0.05em !important;
    padding: 8px 20px !important;
    transition: all 0.2s !important;
}}
.stButton > button:hover {{
    opacity: 0.88 !important;
    transform: translateY(-1px) !important;
    box-shadow: 0 6px 24px {t["primary"]}44 !important;
}}
.stButton > button:active {{
    transform: translateY(0px) !important;
}}

/* ── Inputs ── */
.stSelectbox > div > div,
.stMultiSelect > div > div,
.stNumberInput > div > div > input,
.stTextInput > div > div > input,
.stTextArea textarea {{
    background: var(--surface2) !important;
    border-color: var(--border) !important;
    color: var(--text) !important;
    border-radius: 8px !important;
}}
.stSelectbox > div > div:focus-within,
.stNumberInput > div > div:focus-within,
.stTextInput > div > div:focus-within {{
    box-shadow: {input_shadow} !important;
    border-color: var(--primary) !important;
}}

/* ── Slider ── */
.stSlider [data-baseweb="slider"] [role="slider"] {{
    background-color: var(--primary) !important;
    border-color: var(--primary) !important;
}}
.stSlider [data-baseweb="slider"] div[style*="background"] {{
    background: var(--primary) !important;
}}

/* ── Radio / Checkbox ── */
.stRadio label span, .stCheckbox label span {{
    color: var(--text) !important;
}}

/* ── Dataframe ── */
.stDataFrame {{
    border-radius: 10px;
    overflow: hidden;
    border: 1px solid var(--border);
}}
.stDataFrame thead th {{
    background: var(--surface2) !important;
    color: var(--muted) !important;
    font-family: 'Space Mono', monospace;
    font-size: 0.72rem;
    letter-spacing: 0.06em;
    text-transform: uppercase;
}}

/* ── Expander ── */
.streamlit-expanderHeader {{
    background: var(--surface2) !important;
    border-radius: 8px !important;
    border: 1px solid var(--border) !important;
    color: var(--text) !important;
    font-family: 'Space Mono', monospace;
    font-size: 0.78rem;
    transition: border-color 0.2s;
}}
.streamlit-expanderHeader:hover {{
    border-color: var(--primary) !important;
}}
.streamlit-expanderContent {{
    background: var(--surface) !important;
    border: 1px solid var(--border) !important;
    border-top: none !important;
    border-radius: 0 0 8px 8px !important;
}}

/* ── Alerts ── */
.stAlert {{
    border-radius: 10px !important;
    border-width: 1px !important;
}}
div[data-baseweb="notification"][kind="positive"] {{
    background: {t["success"]}18 !important;
    border-color: {t["success"]}60 !important;
    color: var(--text) !important;
}}
div[data-baseweb="notification"][kind="warning"] {{
    background: {t["secondary"]}18 !important;
    border-color: {t["secondary"]}60 !important;
    color: var(--text) !important;
}}
div[data-baseweb="notification"][kind="negative"] {{
    background: {t["danger"]}18 !important;
    border-color: {t["danger"]}60 !important;
    color: var(--text) !important;
}}
div[data-baseweb="notification"][kind="info"] {{
    background: {t["info"]}18 !important;
    border-color: {t["info"]}60 !important;
    color: var(--text) !important;
}}

/* ── Divider ── */
hr {{ border-color: var(--border) !important; opacity: 1; }}

/* ── Scrollbar ── */
::-webkit-scrollbar {{ width: 6px; height: 6px; }}
::-webkit-scrollbar-track {{ background: {scrollbar_track}; border-radius: 3px; }}
::-webkit-scrollbar-thumb {{ background: {scrollbar_thumb}80; border-radius: 3px; }}
::-webkit-scrollbar-thumb:hover {{ background: {scrollbar_thumb}; }}

/* ── Custom components ── */
.logo-area {{
    <div style="
        font-family: 'Space Mono', monospace;
        font-size: 1.5rem;
        font-weight: 700;
        letter-spacing: -0.02em;
    ">
        <span style="color:var(--primary)">Data</span>
        <span style="color:{'f7c26a'}">Sci</span>
    </div>
}}
.logo-area span {{ color: var(--secondary); }}
.logo-tagline {{
    font-size: 0.7rem;
    color: var(--muted);
    font-family: 'DM Sans', sans-serif;
    margin-top: 2px;
    letter-spacing: 0.03em;
}}

.sec-header {{
    font-family: 'Space Mono', monospace;
    font-size: 0.68rem;
    letter-spacing: 0.2em;
    text-transform: uppercase;
    color: var(--muted);
    margin: 20px 0 10px;
    border-bottom: 1px solid var(--border);
    padding-bottom: 7px;
    display: flex;
    align-items: center;
    gap: 8px;
}}

.badge {{
    display: inline-block;
    background: var(--surface2);
    border: 1px solid var(--border);
    border-radius: 20px;
    padding: 3px 11px;
    font-family: 'Space Mono', monospace;
    font-size: 0.66rem;
    color: var(--accent);
    margin: 2px;
    letter-spacing: 0.03em;
}}
.badge-num {{ color: var(--secondary); border-color: {t["secondary"]}44; }}
.badge-cat {{ color: var(--primary); border-color: {t["primary"]}44; }}
.badge-info {{ color: var(--info); border-color: {t["info"]}44; }}

.log-panel {{
    background: var(--bg);
    border: 1px solid var(--border);
    border-radius: 10px;
    padding: 12px 14px;
    font-family: 'Space Mono', monospace;
    font-size: 0.68rem;
    color: var(--accent);
    max-height: 180px;
    overflow-y: auto;
    line-height: 1.8;
}}

.theme-toggle-label {{
    font-family: 'Space Mono', monospace;
    font-size: 0.68rem;
    color: var(--muted);
    letter-spacing: 0.08em;
    text-transform: uppercase;
    margin-bottom: 4px;
}}

.result-card {{
    border-radius: 12px;
    padding: 18px 22px;
    margin-top: 12px;
    border-width: 1px;
    border-style: solid;
}}
.result-card-label {{
    font-family: 'Space Mono', monospace;
    font-size: 0.68rem;
    letter-spacing: 0.15em;
    text-transform: uppercase;
    margin-bottom: 6px;
}}
.result-card-value {{
    font-size: 1.9rem;
    font-weight: 700;
    color: var(--text);
    line-height: 1.2;
}}
.result-card-sub {{
    font-size: 0.78rem;
    color: var(--muted);
    margin-top: 4px;
}}

.model-banner {{
    border-radius: 12px;
    padding: 16px 20px;
    margin-top: 12px;
}}
.model-banner-label {{
    font-family: 'Space Mono', monospace;
    font-size: 0.72rem;
    letter-spacing: 0.15em;
    text-transform: uppercase;
    margin-bottom: 4px;
}}
.model-banner-name {{
    font-size: 1.35rem;
    font-weight: 700;
    color: var(--text);
}}

.page-title {{
    font-family: 'Space Mono', monospace;
    font-size: 1.5rem;
    font-weight: 700;
    color: var(--text);
    margin-bottom: 4px;
}}
.page-subtitle {{
    color: var(--muted);
    font-size: 0.85rem;
    margin-bottom: 20px;
}}
</style>
""", unsafe_allow_html=True)


def log(msg: str):
    ts = datetime.now().strftime("%H:%M:%S")
    st.session_state.logs.append(f"[{ts}] {msg}")
    if len(st.session_state.logs) > 80:
        st.session_state.logs = st.session_state.logs[-80:]


def style_fig(fig):
    """Apply active theme colors to a matplotlib figure."""
    t, pal, _ = get_theme()
    fig.patch.set_facecolor(t["surface"])
    for ax in fig.get_axes():
        ax.set_facecolor(t["bg"])
        ax.tick_params(colors=t["muted"], labelsize=9)
        ax.xaxis.label.set_color(t["muted"])
        ax.yaxis.label.set_color(t["muted"])
        ax.title.set_color(t["text"])
        for spine in ax.spines.values():
            spine.set_edgecolor(t["border"])
        # Legend
        leg = ax.get_legend()
        if leg:
            leg.get_frame().set_facecolor(t["surface2"])
            leg.get_frame().set_edgecolor(t["border"])
            for txt in leg.get_texts():
                txt.set_color(t["text"])
    return fig


def themed_bar_chart(results_df, metrics_to_plot, title="Model Comparison"):
    t, pal, _ = get_theme()
    fig, ax = plt.subplots(figsize=(max(8, len(results_df) * 2), 5))
    x     = np.arange(len(results_df))
    width = 0.8 / max(len(metrics_to_plot), 1)

    for j, metric in enumerate(metrics_to_plot):
        offset = (j - len(metrics_to_plot) / 2) * width + width / 2
        bars = ax.bar(
            x + offset, results_df[metric], width,
            label=metric, color=pal[j % len(pal)], alpha=0.88,
        )
        for bar in bars:
            h = bar.get_height()
            ax.text(
                bar.get_x() + bar.get_width() / 2, h + 0.005,
                f"{h:.3f}", ha="center", va="bottom",
                fontsize=7, color=t["text"],
            )

    ax.set_xticks(x)
    ax.set_xticklabels(results_df.index, rotation=15, ha="right", color=t["text"])
    ax.set_ylim(0, max(results_df[metrics_to_plot].max().max() * 1.25, 1.15))
    ax.set_ylabel("Score", color=t["muted"])
    ax.set_title(title, color=t["text"], fontsize=11, fontfamily="monospace")
    ax.tick_params(colors=t["muted"])
    return style_fig(fig)


def result_card_html(label, value, sub, bg_color, border_color, label_color):
    return (
        f"<div class='result-card' style='background:{bg_color};border-color:{border_color};'>"
        f"<div class='result-card-label' style='color:{label_color};'>{label}</div>"
        f"<div class='result-card-value'>{value}</div>"
        f"<div class='result-card-sub'>{sub}</div>"
        f"</div>"
    )


def model_banner_html(label, name, bg, border, label_color):
    return (
        f"<div class='model-banner' style='background:{bg};border:1px solid {border};'>"
        f"<div class='model-banner-label' style='color:{label_color};'>{label}</div>"
        f"<div class='model-banner-name'>{name}</div>"
        f"</div>"
    )

Writing ui.py


### EDA functions file

In [112]:
%%writefile EDA.py
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
import warnings
import io
import os
import pickle
import traceback
from datetime import datetime
import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import col, sum, when, mean, trim, lower, to_date, regexp_replace, log1p, variance
from pyspark.sql.types import NumericType
import missingno as msno

# PySpark ML imports
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler, MinMaxScaler, ChiSqSelector
from pyspark.sql.functions import min as spark_min
def get_shape(data):

    return (data.count(), len(data.columns))

def get_n_of_rows(data):

    return data.count()

def get_n_of_columns(data):

    return len(data.columns)

def get_columns(data):

    return data.columns

def get_rows(data, n=10):

    return data.show(n)

def get_describe(data):

    return data.describe().show()

def get_schema(data):

    return data.printSchema()

# ─── Visualization Functions ──────────────────────────────────────────────────
def histogram_plot(data, column, sample_fraction=0.05):

    df = data.select(column).dropna().sample(False, sample_fraction).toPandas()
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.histplot(df[column], bins=40, kde=False, color='#7c6af7', ax=ax)
    ax.set_title(f"Histogram of {column}", color='#e8e8f0')
    ax.set_xlabel(column, color='#888899')
    ax.set_ylabel("Frequency", color='#888899')
    return fig

def kde_plot(data, column, sample_fraction=0.05):

    df = data.select(column).dropna().sample(False, sample_fraction).toPandas()
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.kdeplot(df[column], fill=True, color='#6af7c2', ax=ax)
    ax.set_title(f"KDE Plot of {column}", color='#e8e8f0')
    ax.set_xlabel(column, color='#888899')
    ax.set_ylabel("Density", color='#888899')
    return fig

def box_plot(data, column, sample_fraction=0.05):

    df = data.select(column).dropna().sample(False, sample_fraction).toPandas()
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.boxplot(x=df[column], color='#f7c26a', ax=ax)
    ax.set_title(f"Box Plot of {column}", color='#e8e8f0')
    ax.set_xlabel(column, color='#888899')
    return fig

def countplot(data, column, top_n=20):

    pdf = (data.groupBy(column)
           .count()
           .orderBy("count", ascending=False)
           .limit(top_n)
           .toPandas())

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.bar(pdf[column].astype(str), pdf['count'], color='#7c6af7')
    ax.set_xticklabels(pdf[column].astype(str), rotation=45, ha='right')
    ax.set_title(f"Countplot - {column}", color='#e8e8f0')
    ax.set_xlabel(column, color='#888899')
    ax.set_ylabel("Count", color='#888899')
    ax.tick_params(colors='#888899')
    return fig

def to_pandas_safe(df, sample_size=0.1):

    return df.sample(fraction=sample_size).toPandas()

def plot_heatmap(df, sample_size=0.2):

    df_pd = to_pandas_safe(df, sample_size)
    corr = df_pd.corr(numeric_only=True)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, ax=ax,
                fmt='.2f', annot_kws={'color': '#e8e8f0'})
    ax.set_title("Correlation Heatmap", color='#e8e8f0')
    ax.tick_params(colors='#888899')
    return fig

def plot_scatter(df, x_col, y_col, sample_size=0.1):

    df_pd = to_pandas_safe(df, sample_size)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.scatterplot(x=x_col, y=y_col, data=df_pd, color='#7c6af7', alpha=0.6, ax=ax)
    ax.set_title(f"{x_col} vs {y_col}", color='#e8e8f0')
    ax.set_xlabel(x_col, color='#888899')
    ax.set_ylabel(y_col, color='#888899')
    ax.tick_params(colors='#888899')
    return fig

def plot_pair(df, sample_size=0.1):

    df_pd = to_pandas_safe(df, sample_size)
    df_pd = df_pd.select_dtypes(include="number")


    if len(df_pd.columns) > 5:
        df_pd = df_pd.iloc[:, :5]

    g = sns.pairplot(df_pd, diag_kind='kde', plot_kws={'alpha': 0.5, 'color': '#7c6af7'})
    return g.fig

def plot_missing(df, sample_size=0.3):

    df_pd = to_pandas_safe(df, sample_size)
    fig, ax = plt.subplots(figsize=(10, 5))
    msno.matrix(df_pd, ax=ax, color=(0.486, 0.416, 0.969), sparkline=False)
    ax.set_title("Missing Values Matrix", color='#e8e8f0')
    ax.tick_params(colors='#888899')
    return fig

def plot_violin(df, x_col, y_col, sample_size=0.1):

    df_pd = to_pandas_safe(df, sample_size)
    fig, ax = plt.subplots(figsize=(10, 6))


    top_categories = (df_pd[x_col].value_counts().head(20).index)
    df_filtered = df_pd[df_pd[x_col].isin(top_categories)]

    sns.violinplot(x=x_col, y=y_col, data=df_filtered, palette='viridis', ax=ax)
    ax.set_title(f"{y_col} distribution by {x_col}", color='#e8e8f0')
    ax.set_xlabel(x_col, color='#888899')
    ax.set_ylabel(y_col, color='#888899')
    ax.tick_params(colors='#888899', axis='x', rotation=45)
    return fig

def plot_stacked_bar(df, index_col, cols, stacked=True, sample_size=0.2):

    df_pd = to_pandas_safe(df, sample_size)


    top_categories = (df_pd[index_col].value_counts().head(20).index)
    df_filtered = df_pd[df_pd[index_col].isin(top_categories)]

    df_plot = df_filtered.groupby(index_col)[cols].mean()

    fig, ax = plt.subplots(figsize=(10, 6))
    df_plot.plot(kind='bar', stacked=stacked, ax=ax, colormap='viridis')
    ax.set_title("Stacked Bar Chart" if stacked else "Grouped Bar Chart", color='#e8e8f0')
    ax.set_ylabel("Values", color='#888899')
    ax.set_xlabel(index_col, color='#888899')
    ax.tick_params(colors='#888899', axis='x', rotation=45)
    ax.legend(loc='best', facecolor='#111118', labelcolor='#e8e8f0')
    return fig

# ─── Missing Detection ────────────────────────────────────────────────────────
def check_missing_values(df, extra_missing=None):

    if extra_missing is None:
        extra_missing = ["na", "null", "n/a"]

    df_cached = df.cache()
    total_rows = df_cached.count()

    missing_exprs = []
    for c in df.columns:
        cond = (
            col(c).isNull() |
            (trim(col(c)) == "") |
            (lower(trim(col(c))).isin(extra_missing))
        )
        missing_exprs.append(sum(when(cond, 1).otherwise(0)).alias(c))

    missing_df = df_cached.select(missing_exprs)

    missing_df = missing_df.selectExpr(
        "stack({0}, {1}) as (column_name, missing_count)".format(
            len(df.columns),
            ", ".join([f"'{c}', {c}" for c in df.columns])
        )
    )

    missing_df = missing_df.withColumn(
        "missing_percentage",
        (col("missing_count") / total_rows) * 100
    )

    return missing_df.orderBy(col("missing_count").desc())

# ─── Duplicates Detection ─────────────────────────────────────────────────────
def check_duplicates(df, subset=None):

    df_cached = df.cache()
    total_rows = df_cached.count()

    if subset:
        grouped = df_cached.groupBy(subset).count()
    else:
        grouped = df_cached.groupBy(df.columns).count()

    duplicates_df = grouped.filter(col("count") > 1)
    duplicate_rows = duplicates_df.selectExpr("sum(count - 1) as dup").collect()[0]["dup"]

    if duplicate_rows is None:
        duplicate_rows = 0

    duplicate_percentage = (duplicate_rows / total_rows) * 100 if total_rows > 0 else 0

    result = df.sparkSession.createDataFrame([
        (total_rows, duplicate_rows, duplicate_percentage)
    ], ["total_rows", "duplicate_rows", "duplicate_percentage"])

    return result

def get_duplicate_groups(df, subset=None):

    if subset is None:
        subset = df.columns

    duplicates = (df.groupBy(subset)
                  .count()
                  .filter(col("count") > 1))
    return duplicates

# ─── Outlier Detection ────────────────────────────────────────────────────────
def iqr_outlier_detection(data, column):

    q1, q3 = data.approxQuantile(column, [0.25, 0.75], 0.01)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outliers = data.filter(
        (data[column] < lower_bound) | (data[column] > upper_bound))
    outliers_number = outliers.count()

    return outliers, outliers_number, lower_bound, upper_bound

def detect_outliers_zscore(data, column, threshold=3):

    stats = data.select(
        F.mean(column).alias("mean"),
        F.stddev(column).alias("std")
    ).collect()[0]

    mean_val = stats["mean"]
    std_val = stats["std"]

    outliers = data.filter(
        F.abs((F.col(column) - mean_val) / std_val) > threshold
    )
    outliers_number = outliers.count()

    return outliers, outliers_number, mean_val, std_val

def detect_outliers_percentile(data, column, lower_percentile=0.01, upper_percentile=0.99):

    bounds = data.approxQuantile(column, [lower_percentile, upper_percentile], 0.01)
    lower = bounds[0]
    upper = bounds[1]

    outliers = data.filter((data[column] < lower) | (data[column] > upper))
    outliers_number = outliers.count()

    return outliers, outliers_number, lower, upper

Overwriting EDA.py


### preprocessing functions file

In [113]:
%%writefile preprocessing.py
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
import warnings
import io
import os
import pickle
import traceback
from datetime import datetime
import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import col, sum, when, mean, trim, lower, to_date, regexp_replace, log1p, variance
from pyspark.sql.types import NumericType
import missingno as msno

# PySpark ML imports
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler, MinMaxScaler, ChiSqSelector
from pyspark.sql.functions import min as spark_min
def count_duplicates_spark(df):

    total_rows = df.count()
    unique_rows = df.dropDuplicates().count()
    duplicates = total_rows - unique_rows
    return duplicates, total_rows, unique_rows

def remove_duplicates_spark(df):

    cleaned_df = df.dropDuplicates()
    return cleaned_df

def drop_rows_missing(df):

    return df.dropna()

def drop_rows_missing_cols(df, cols):

    return df.dropna(subset=cols)

def impute_mean(df, cols):

    for c in cols:
        mean_value = df.select(mean(c)).collect()[0][0]
        df = df.fillna({c: mean_value})
    return df

def impute_median(df, cols):

    for c in cols:
        median_value = df.approxQuantile(c, [0.5], 0.01)[0]
        df = df.fillna({c: median_value})
    return df

def impute_mode(df, cols):

    for c in cols:
        mode_value = df.groupBy(c).count().orderBy("count", ascending=False).first()[0]
        df = df.fillna({c: mode_value})
    return df

def forward_fill(df, target_col, order_col):

    w = Window.orderBy(order_col).rowsBetween(Window.unboundedPreceding, 0)
    df_filled = df.withColumn(target_col, F.last(target_col, ignorenulls=True).over(w))
    return df_filled

def backward_fill(df, target_col, order_col):

    w = Window.orderBy(order_col).rowsBetween(0, Window.unboundedFollowing)
    df_filled = df.withColumn(target_col, F.first(target_col, ignorenulls=True).over(w))
    return df_filled

def handle_inconsistent_typo(data, column, rare_threshold=5):

    data = data.withColumn(column, lower(trim(col(column))))
    unique_values = data.select(column).distinct()
    freq_table = data.groupBy(column).count().orderBy("count")
    suspicious = freq_table.filter(col("count") <= rare_threshold)
    return data, unique_values, freq_table, suspicious

def handle_invalid_min(data, column, min_val):

    invalid_rows = data.filter(col(column) <= min_val)
    return {
        "invalid_rows": invalid_rows,
        "invalid_count": invalid_rows.count(),
        'filtered_data_set': data.filter(col(column) > min_val)
    }

def check_unique_values(df, col_name):

    return df.select(col_name).distinct()

def clean_text_columns(df, cols):

    for c in cols:
        df = df.withColumn(c, lower(trim(col(c))))
    return df

def standardize_date(df, col_name, format="yyyy-MM-dd"):

    return df.withColumn(col_name, to_date(col(col_name), format))

def clean_numeric_text(df, col_name):

    df = df.withColumn(col_name, regexp_replace(col(col_name), "[^0-9.]", ""))
    return df

def split_data_spark(df, test_size=0.2, seed=42):

    train_df, test_df = df.randomSplit([1 - test_size, test_size], seed=seed)
    return train_df, test_df

def detect_columns(df, exclude_cols=[]):

    numeric_cols = [
        c for c, t in df.dtypes
        if t in ('int', 'double', 'float', 'long', 'bigint', 'decimal')
        and c not in exclude_cols
    ]
    categorical_cols = [
        c for c, t in df.dtypes
        if t == "string"
        and c not in exclude_cols
    ]
    return numeric_cols, categorical_cols

def apply_onehot_encoding_spark(train_df, test_df, col_name):

    indexer = StringIndexer(inputCol=col_name, outputCol=col_name + "_indexed", handleInvalid="keep")
    index_model = indexer.fit(train_df)
    train_indexed = index_model.transform(train_df)
    test_indexed = index_model.transform(test_df)
    encoder = OneHotEncoder(inputCols=[col_name + "_indexed"], outputCols=[col_name + "_ohe"])
    encoder_model = encoder.fit(train_indexed)
    train_encoded = encoder_model.transform(train_indexed)
    test_encoded = encoder_model.transform(test_indexed)
    return train_encoded, test_encoded

def apply_label_encoding_spark(train_df, test_df, col_name):

    indexer = StringIndexer(inputCol=col_name, outputCol=col_name + "_indexed", handleInvalid="keep")
    model = indexer.fit(train_df)
    train_encoded = model.transform(train_df)
    test_encoded = model.transform(test_df)
    return train_encoded, test_encoded

def frequency_encoding(train_df, test_df, cols):

    if len(cols) == 0:
        return train_df, test_df
    total = train_df.count()
    for c in cols:
        freq_df = train_df.groupBy(c).count()
        freq_df = freq_df.withColumn(c + "_freq", F.col("count") / total).select(c, c + "_freq")
        train_df = train_df.join(freq_df, on=c, how="left")
        test_df = test_df.join(freq_df, on=c, how="left")
        train_df = train_df.withColumnRenamed(c + "_freq", c).drop(c)
        test_df = test_df.withColumnRenamed(c + "_freq", c).drop(c)
    return train_df, test_df

def apply_standard_scaler_spark(train_df, test_df):

    numeric_cols = [col_name for col_name, col_type in train_df.dtypes if col_type in ('int', 'double', 'float', 'long', 'bigint', 'decimal')]
    if not numeric_cols:
        return train_df, test_df
    assembler = VectorAssembler(inputCols=numeric_cols, outputCol="features_vec")
    train_df = assembler.transform(train_df)
    test_df = assembler.transform(test_df)
    scaler = StandardScaler(inputCol="features_vec", outputCol="scaled_features", withMean=True, withStd=True)
    scaler_model = scaler.fit(train_df)
    train_scaled = scaler_model.transform(train_df)
    test_scaled = scaler_model.transform(test_df)
    return train_scaled, test_scaled

def apply_minmax_scaler_spark(train_df, test_df):

    numeric_cols = [col_name for col_name, col_type in train_df.dtypes if col_type in ('int', 'double', 'float', 'long', 'bigint', 'decimal')]
    if not numeric_cols:
        return train_df, test_df
    assembler = VectorAssembler(inputCols=numeric_cols, outputCol="features_vec")
    train_df = assembler.transform(train_df)
    test_df = assembler.transform(test_df)
    scaler = MinMaxScaler(inputCol="features_vec", outputCol="scaled_features")
    scaler_model = scaler.fit(train_df)
    train_scaled = scaler_model.transform(train_df)
    test_scaled = scaler_model.transform(test_df)
    return train_scaled, test_scaled

def select_features_variance_spark(train_df, threshold=0.01):

    numeric_cols = [c for c, t in train_df.dtypes if t in ('int', 'double', 'float', 'long', 'bigint', 'decimal')]
    variances = train_df.select([variance(col(c)).alias(c) for c in numeric_cols]).collect()[0].asDict()
    selected_features = [col_name for col_name, var_value in variances.items() if var_value is not None and var_value > threshold]
    return selected_features

def select_features_chisq_spark(train_df, label_col, k=7):

    numeric_cols = [c for c, t in train_df.dtypes if t in ('int', 'double', 'float', 'long', 'bigint', 'decimal')]
    feature_cols = [c for c in numeric_cols if c != label_col and train_df.select(spark_min(col(c))).collect()[0][0] >= 0]
    if len(feature_cols) < 2:
        return feature_cols
    assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_vec")
    train_vec = assembler.transform(train_df)
    selector = ChiSqSelector(numTopFeatures=min(k, len(feature_cols)), featuresCol="features_vec", outputCol="selected_features", labelCol=label_col)
    model = selector.fit(train_vec)
    selected_indices = model.selectedFeatures
    selected_features = [feature_cols[i] for i in selected_indices if i < len(feature_cols)]
    return selected_features

Overwriting preprocessing.py


### modelling functions file

In [114]:
%%writefile models.py

import findspark
findspark.init()

from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import (
    LogisticRegression,
    RandomForestClassifier,
    GBTClassifier,
    DecisionTreeClassifier,
    LinearSVC,
)
from pyspark.ml.regression import (
    LinearRegression,
    DecisionTreeRegressor,
    RandomForestRegressor,
    GBTRegressor,
    GeneralizedLinearRegression,
    IsotonicRegression,
)
from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator,
    BinaryClassificationEvaluator,
    RegressionEvaluator,
)

# ── XGBoost ────────────────────────────────────────────────────────────────────
try:
    from xgboost.spark import SparkXGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    try:
        from sparkxgb import XGBoostClassifier as SparkXGBClassifier
        XGBOOST_AVAILABLE = True
    except ImportError:
        XGBOOST_AVAILABLE = False


# ==============================================================================
# CLASSIFICATION
# ==============================================================================

def detect_task_type(df, target_col):
    """Returns 'binary' or 'multiclass'."""
    unique_count = df.select(target_col).distinct().count()
    return "binary" if unique_count == 2 else "multiclass"


def get_available_classification_models():
    models = [
        "Logistic Regression",
        "Random Forest",
        "GBT",
        "Decision Tree",
        "SVM (Linear)",
    ]
    if XGBOOST_AVAILABLE:
        models.append("XGBoost")
    return models


def train_logistic_regression(train_df, label_col, task_type="binary"):
    params = dict(labelCol=label_col, featuresCol="features", maxIter=100)
    if task_type == "multiclass":
        params["family"] = "multinomial"
    return LogisticRegression(**params).fit(train_df)


def train_random_forest_classifier(train_df, label_col, task_type="binary"):
    return RandomForestClassifier(
        labelCol=label_col,
        featuresCol="features",
        numTrees=100,
    ).fit(train_df)


def train_gbt_classifier(train_df, label_col, task_type="binary"):
    return GBTClassifier(
        labelCol=label_col,
        featuresCol="features",
        maxIter=50,
    ).fit(train_df)


def train_decision_tree_classifier(train_df, label_col, task_type="binary"):
    return DecisionTreeClassifier(
        labelCol=label_col,
        featuresCol="features",
    ).fit(train_df)


def train_svm(train_df, label_col, task_type="binary"):
    svc = LinearSVC(labelCol=label_col, featuresCol="features", maxIter=100)
    if task_type == "multiclass":
        from pyspark.ml.classification import OneVsRest
        ovr = OneVsRest(classifier=svc, labelCol=label_col, featuresCol="features")
        return ovr.fit(train_df)
    return svc.fit(train_df)


def train_xgboost(train_df, label_col, task_type="binary"):
    if not XGBOOST_AVAILABLE:
        raise ImportError("XGBoost not available. Run: pip install xgboost[spark]")
    num_class = train_df.select(label_col).distinct().count()
    params = dict(
        label_col=label_col,
        features_col="features",
        num_round=100,
        max_depth=6,
        eta=0.1,
    )
    if task_type == "multiclass":
        params["objective"] = "multi:softprob"
        params["num_class"] = num_class
    else:
        params["objective"] = "binary:logistic"
    return SparkXGBClassifier(**params).fit(train_df)


def train_classification_model(model_name: str, train_df, label_col: str, task_type: str):
    dispatch = {
        "Logistic Regression": train_logistic_regression,
        "Random Forest":       train_random_forest_classifier,
        "GBT":                 train_gbt_classifier,
        "Decision Tree":       train_decision_tree_classifier,
        "SVM (Linear)":        train_svm,
        "XGBoost":             train_xgboost,
    }
    if model_name not in dispatch:
        raise ValueError(f"Unknown classification model: {model_name}")
    return dispatch[model_name](train_df, label_col, task_type)


def evaluate_classification_model(model, test_df, label_col: str, task_type: str) -> dict:
    predictions = model.transform(test_df)
    mc_eval = MulticlassClassificationEvaluator(labelCol=label_col)

    def mc(metric):
        return mc_eval.evaluate(predictions, {mc_eval.metricName: metric})

    results = {
        "Accuracy":  mc("accuracy"),
        "F1":        mc("f1"),
        "Precision": mc("weightedPrecision"),
        "Recall":    mc("weightedRecall"),
    }

    if task_type == "binary":
        try:
            bin_eval = BinaryClassificationEvaluator(
                labelCol=label_col,
                rawPredictionCol="rawPrediction",
            )
            results["AUC-ROC"] = bin_eval.evaluate(
                predictions, {bin_eval.metricName: "areaUnderROC"}
            )
            results["AUC-PR"] = bin_eval.evaluate(
                predictions, {bin_eval.metricName: "areaUnderPR"}
            )
        except Exception:
            pass

    return results


def select_best_model(results: dict, primary_metric: str = "F1"):
    best = max(results, key=lambda n: results[n].get(primary_metric, 0))
    return best, results[best]


# ==============================================================================
# REGRESSION
# ==============================================================================

def get_available_regression_models():
    return [
        "Linear Regression",
        "Decision Tree Regressor",
        "Random Forest Regressor",
        "GBT Regressor",
        "Generalized Linear Regression",
        "Isotonic Regression",
    ]


def train_linear_regression(train_df, label_col):
    return LinearRegression(
        labelCol=label_col,
        featuresCol="features",
        maxIter=100,
        regParam=0.1,
        elasticNetParam=0.0,
    ).fit(train_df)


def train_decision_tree_regressor(train_df, label_col):
    return DecisionTreeRegressor(
        labelCol=label_col,
        featuresCol="features",
        maxDepth=5,
    ).fit(train_df)


def train_random_forest_regressor(train_df, label_col):
    return RandomForestRegressor(
        labelCol=label_col,
        featuresCol="features",
        numTrees=100,
        maxDepth=5,
    ).fit(train_df)


def train_gbt_regressor(train_df, label_col):
    return GBTRegressor(
        labelCol=label_col,
        featuresCol="features",
        maxIter=50,
        maxDepth=5,
    ).fit(train_df)


def train_generalized_linear_regression(train_df, label_col):
    return GeneralizedLinearRegression(
        labelCol=label_col,
        featuresCol="features",
        family="gaussian",
        link="identity",
        maxIter=100,
    ).fit(train_df)


def train_isotonic_regression(train_df, label_col):
    # IsotonicRegression requires a single feature column
    return IsotonicRegression(
        labelCol=label_col,
        featuresCol="features",
        isotonic=True,
    ).fit(train_df)


def train_regression_model(model_name: str, train_df, label_col: str):
    dispatch = {
        "Linear Regression":              train_linear_regression,
        "Decision Tree Regressor":        train_decision_tree_regressor,
        "Random Forest Regressor":        train_random_forest_regressor,
        "GBT Regressor":                  train_gbt_regressor,
        "Generalized Linear Regression":  train_generalized_linear_regression,
        "Isotonic Regression":            train_isotonic_regression,
    }
    if model_name not in dispatch:
        raise ValueError(f"Unknown regression model: {model_name}")
    return dispatch[model_name](train_df, label_col)


def evaluate_regression_model(model, test_df, label_col: str) -> dict:
    predictions = model.transform(test_df)
    evaluator = RegressionEvaluator(labelCol=label_col, predictionCol="prediction")

    def reg(metric):
        return evaluator.evaluate(predictions, {evaluator.metricName: metric})

    results = {
        "RMSE":  reg("rmse"),
        "MAE":   reg("mae"),
        "R2":    reg("r2"),
        "MSE":   reg("mse"),
    }
    return results


def select_best_regression_model(results: dict, primary_metric: str = "R2"):
    # For regression, higher R2 is better, lower RMSE/MAE/MSE is better
    if primary_metric == "R2":
        best = max(results, key=lambda n: results[n].get(primary_metric, float('-inf')))
    else:
        best = min(results, key=lambda n: results[n].get(primary_metric, float('inf')))
    return best, results[best]

Overwriting models.py


### visualization_tab file

In [115]:
%%writefile visualization_tab.py
import streamlit as st
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import warnings
warnings.filterwarnings('ignore')

from pyspark.sql import functions as F

from EDA import *
from preprocessing import *
from ui import get_theme, style_fig


def visualization_tab(df):
    t, pal, _ = get_theme()

    st.markdown('<div class="sec-header">📊 Visualization Dashboard</div>', unsafe_allow_html=True)

    numeric_cols = [c for c, dtype in df.dtypes if any(nt in dtype.lower()
                   for nt in ['int', 'double', 'float', 'decimal', 'long', 'short'])]
    categorical_cols = [c for c in df.columns if c not in numeric_cols]
    all_cols = df.columns

    viz_subtab = st.tabs([
        "📈 Distribution", "📊 Categorical",
        "🔗 Relationships", "🎻 Advanced", "📉 Missing & Correlation",
    ])

    st.sidebar.markdown("### 🎛️ Visualization Settings")
    sample_size = st.sidebar.slider("Sample Size", 0.01, 1.0, 0.1, 0.01, key="viz_sample")
    st.sidebar.markdown("---")

    # ── Distribution ──
    with viz_subtab[0]:
        st.markdown("#### 📈 Distribution Plots")
        col1, col2 = st.columns(2)

        with col1:
            num_col = st.selectbox("Select Column", numeric_cols if numeric_cols else all_cols, key="dist_col")
            plot_type = st.radio("Plot Type", ["Histogram", "KDE Plot", "Box Plot"], horizontal=True, key="dist_type")

            if st.button("Generate Distribution Plot", key="btn_dist"):
                with st.spinner("Generating plot..."):
                    if plot_type == "Histogram":
                        fig = histogram_plot(df, num_col, sample_size)
                    elif plot_type == "KDE Plot":
                        fig = kde_plot(df, num_col, sample_size)
                    else:
                        fig = box_plot(df, num_col, sample_size)
                    st.pyplot(style_fig(fig)); plt.close()

        with col2:
            st.markdown("##### 📋 Statistical Summary")
            if numeric_cols:
                stats_df = df.select(
                    F.mean(num_col).alias("Mean"),
                    F.stddev(num_col).alias("Std Dev"),
                    F.min(num_col).alias("Min"),
                    F.expr(f"percentile_approx({num_col}, 0.25)").alias("Q1"),
                    F.expr(f"percentile_approx({num_col}, 0.5)").alias("Median"),
                    F.expr(f"percentile_approx({num_col}, 0.75)").alias("Q3"),
                    F.max(num_col).alias("Max"),
                ).toPandas()
                for cn in stats_df.columns:
                    val = stats_df[cn].iloc[0]
                    if val is not None:
                        st.metric(cn, f"{val:.4f}")

    # ── Categorical ──
    with viz_subtab[1]:
        st.markdown("#### 📊 Categorical Analysis")
        if categorical_cols:
            col1, col2 = st.columns(2)
            with col1:
                cat_col = st.selectbox("Select Categorical Column", categorical_cols, key="cat_col")
                top_n = st.slider("Top Categories", 5, 50, 20, 5, key="top_n")
                if st.button("Generate Count Plot", key="btn_cat"):
                    with st.spinner("Generating..."):
                        fig = countplot(df, cat_col, top_n)
                        st.pyplot(style_fig(fig)); plt.close()
            with col2:
                st.markdown("##### 📋 Frequency Table")
                freq_df = (df.groupBy(cat_col).count()
                           .orderBy(F.col("count").desc()).limit(top_n).toPandas())
                st.dataframe(freq_df, use_container_width=True)
        else:
            st.info("No categorical columns found in the dataset.")

    # ── Relationships ──
    with viz_subtab[2]:
        st.markdown("#### 🔗 Relationship Analysis")
        if len(numeric_cols) >= 2:
            col1, col2 = st.columns(2)
            with col1:
                x_col = st.selectbox("X Axis", numeric_cols, key="scatter_x")
                y_col = st.selectbox("Y Axis", [c for c in numeric_cols if c != x_col], key="scatter_y")
                if st.button("Generate Scatter Plot", key="btn_scatter"):
                    with st.spinner("Generating..."):
                        fig = plot_scatter(df, x_col, y_col, sample_size)
                        st.pyplot(style_fig(fig)); plt.close()
            with col2:
                st.markdown("##### 📊 Correlation")
                corr_val = df.stat.corr(x_col, y_col)
                st.metric(f"Pearson r ({x_col} vs {y_col})", f"{corr_val:.4f}")
                st.caption("±1 → strong | 0 → none")

            st.markdown("---")
            st.markdown("#### Pair Plot Matrix")
            if len(numeric_cols) <= 6:
                if st.button("Generate Pair Plot", key="btn_pair"):
                    with st.spinner("Generating (may take a moment)..."):
                        fig = plot_pair(df, min(sample_size, 0.2))
                        st.pyplot(style_fig(fig)); plt.close()
            else:
                st.info(f"Pair plot skipped: {len(numeric_cols)} numeric columns (max 6).")
        else:
            st.warning("Need at least 2 numeric columns.")

    # ── Advanced ──
    with viz_subtab[3]:
        st.markdown("#### 🎻 Advanced Visualizations")
        adv_type = st.selectbox("Type", ["Violin Plot", "Stacked Bar Chart"], key="adv_type")

        if adv_type == "Violin Plot":
            if categorical_cols and numeric_cols:
                col1, col2 = st.columns(2)
                with col1:
                    x_violin = st.selectbox("Category (X)", categorical_cols, key="violin_x")
                    y_violin = st.selectbox("Value (Y)", numeric_cols, key="violin_y")
                    if st.button("Generate Violin Plot", key="btn_violin"):
                        with st.spinner("Generating..."):
                            fig = plot_violin(df, x_violin, y_violin, sample_size)
                            st.pyplot(style_fig(fig)); plt.close()
                with col2:
                    st.markdown("##### Group Statistics")
                    stats_by_cat = (df.groupBy(x_violin)
                                   .agg(F.mean(y_violin).alias("Mean"),
                                        F.stddev(y_violin).alias("Std Dev"),
                                        F.count(y_violin).alias("Count"))
                                   .orderBy(F.col("Count").desc()).limit(10).toPandas())
                    st.dataframe(stats_by_cat, use_container_width=True)
            else:
                st.warning("Need both categorical and numeric columns.")
        else:
            if categorical_cols and len(numeric_cols) >= 2:
                col1, col2 = st.columns(2)
                with col1:
                    index_col = st.selectbox("Index Column", categorical_cols, key="stacked_index")
                    val_cols  = st.multiselect("Value Columns", numeric_cols,
                                               default=numeric_cols[:min(3, len(numeric_cols))], key="stacked_vals")
                    stacked   = st.checkbox("Stacked", value=True, key="is_stacked")
                    if val_cols and st.button("Generate Stacked Bar Chart", key="btn_stacked"):
                        with st.spinner("Generating..."):
                            fig = plot_stacked_bar(df, index_col, val_cols, stacked, sample_size)
                            st.pyplot(style_fig(fig)); plt.close()
                with col2:
                    if val_cols:
                        preview_df = (df.groupBy(index_col)
                                     .agg(*[F.mean(c).alias(c) for c in val_cols])
                                     .orderBy(F.col(val_cols[0]).desc()).limit(10).toPandas())
                        st.dataframe(preview_df, use_container_width=True)
            else:
                st.warning("Need at least one categorical and 2+ numeric columns.")

    # ── Missing & Correlation ──
    with viz_subtab[4]:
        st.markdown("#### 📉 Data Quality Analysis")
        col1, col2 = st.columns(2)

        with col1:
            st.markdown("##### 🔍 Missing Values")
            missing_df  = check_missing_values(df).toPandas()
            missing_cols_df = missing_df[missing_df['missing_count'] > 0]
            if not missing_cols_df.empty:
                st.dataframe(missing_cols_df, use_container_width=True)
                if st.button("Show Missing Values Chart", key="btn_missing"):
                    with st.spinner("Generating..."):
                        fig, ax = plt.subplots(figsize=(8, 4))
                        mc_sorted = missing_cols_df.sort_values('missing_count', ascending=False)
                        ax.barh(mc_sorted['column_name'], mc_sorted['missing_count'], color=t["danger"])
                        ax.set_title('Missing Values per Column', color=t["text"])
                        ax.set_xlabel('Count', color=t["muted"])
                        st.pyplot(style_fig(fig)); plt.close()
            else:
                st.success("✅ No missing values found!")

        with col2:
            st.markdown("##### 🔥 Correlation Heatmap")
            if len(numeric_cols) >= 2:
                if st.button("Generate Correlation Heatmap", key="btn_heatmap"):
                    with st.spinner("Generating..."):
                        fig = plot_heatmap(df, min(sample_size, 0.2))
                        st.pyplot(style_fig(fig)); plt.close()
            else:
                st.info("Need at least 2 numeric columns.")

Writing visualization_tab.py


### EDA_tab file

In [116]:
%%writefile EDA_tab.py
import streamlit as st
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import warnings
warnings.filterwarnings('ignore')

from pyspark.sql import functions as F

from EDA import *
from preprocessing import *
from ui import get_theme, style_fig


def eda_tab(df, selected_col, col_type):
    t, pal, _ = get_theme()

    st.markdown('<div class="sec-header">Dataset Overview</div>', unsafe_allow_html=True)

    rows, cols_count = get_shape(df)
    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Rows", f"{rows:,}")
    c2.metric("Columns", f"{cols_count:,}")

    missing_df   = check_missing_values(df).toPandas()
    total_missing = missing_df['missing_count'].sum() if not missing_df.empty else 0
    c3.metric("Null Values", f"{int(total_missing):,}")

    dup_result     = check_duplicates(df).toPandas()
    duplicate_rows = dup_result['duplicate_rows'].iloc[0] if not dup_result.empty else 0
    c4.metric("Duplicates", f"{int(duplicate_rows):,}")

    with st.expander("📋 Data Preview", expanded=False):
        st.dataframe(df.limit(100).toPandas(), use_container_width=True)

    with st.expander("📝 Schema & Info", expanded=False):
        schema_data = [{"Column": cn, "Type": ct} for cn, ct in df.dtypes]
        import pandas as pd
        st.dataframe(pd.DataFrame(schema_data), use_container_width=True)

    st.markdown('<div class="sec-header">Column Analysis</div>', unsafe_allow_html=True)

    if not selected_col:
        st.info("Select a column in the sidebar.")
        return

    badge_cls = "badge-num" if col_type == "numeric" else "badge-cat"
    st.markdown(
        f"**Column:** `{selected_col}` &nbsp;"
        f"<span class='badge {badge_cls}'>{col_type}</span>",
        unsafe_allow_html=True,
    )

    if col_type == "numeric":
        stats = df.select(
            F.mean(selected_col).alias("mean"),
            F.stddev(selected_col).alias("std"),
            F.min(selected_col).alias("min"),
            F.max(selected_col).alias("max"),
        ).collect()[0]

        c1, c2, c3, c4 = st.columns(4)
        c1.metric("Mean", f"{stats['mean']:.4f}" if stats['mean'] else "N/A")
        c2.metric("Std",  f"{stats['std']:.4f}"  if stats['std']  else "N/A")
        c3.metric("Min",  f"{stats['min']:.4f}"  if stats['min']  else "N/A")
        c4.metric("Max",  f"{stats['max']:.4f}"  if stats['max']  else "N/A")

        col_a, col_b = st.columns(2)
        with col_a:
            fig = histogram_plot(df, selected_col)
            st.pyplot(style_fig(fig)); plt.close()
        with col_b:
            fig = box_plot(df, selected_col)
            st.pyplot(style_fig(fig)); plt.close()

        with st.expander("🔍 Outlier Detection", expanded=False):
            outlier_method = st.selectbox("Method", ["IQR", "Z-Score", "Percentile"], key="out_meth")
            if outlier_method == "IQR":
                outliers, out_count, lower, upper = iqr_outlier_detection(df, selected_col)
                st.metric("Lower Bound", f"{lower:.4f}")
                st.metric("Upper Bound", f"{upper:.4f}")
                st.metric("Outlier Count", f"{out_count:,}")
                if out_count > 0 and st.button("Show Sample"):
                    st.dataframe(outliers.limit(10).toPandas(), use_container_width=True)

            elif outlier_method == "Z-Score":
                threshold = st.slider("Z-Score Threshold", 1.0, 5.0, 3.0, 0.5)
                outliers, out_count, mean_val, std_val = detect_outliers_zscore(df, selected_col, threshold)
                st.metric("Mean", f"{mean_val:.4f}")
                st.metric("Std", f"{std_val:.4f}")
                st.metric("Outlier Count", f"{out_count:,}")
                if out_count > 0 and st.button("Show Sample"):
                    st.dataframe(outliers.limit(10).toPandas(), use_container_width=True)

            else:
                lower_pct = st.slider("Lower Percentile", 0.01, 0.10, 0.01, 0.01)
                upper_pct = st.slider("Upper Percentile", 0.90, 0.99, 0.99, 0.01)
                outliers, out_count, lower, upper = detect_outliers_percentile(df, selected_col, lower_pct, upper_pct)
                st.metric(f"Lower ({lower_pct*100:.0f}%)", f"{lower:.4f}")
                st.metric(f"Upper ({upper_pct*100:.0f}%)", f"{upper:.4f}")
                st.metric("Outlier Count", f"{out_count:,}")
                if out_count > 0 and st.button("Show Sample"):
                    st.dataframe(outliers.limit(10).toPandas(), use_container_width=True)
    else:
        col_a, col_b = st.columns(2)
        with col_a:
            fig = countplot(df, selected_col)
            st.pyplot(style_fig(fig)); plt.close()
        with col_b:
            freq_df = (df.groupBy(selected_col).count()
                      .orderBy(F.col("count").desc()).limit(20).toPandas())
            st.dataframe(freq_df, use_container_width=True)

Writing EDA_tab.py


### processing_tab file

In [117]:
%%writefile processing_tab.py
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import warnings
warnings.filterwarnings('ignore')

from pyspark.sql import functions as F
from pyspark.sql.functions import col, mean, log1p

from EDA import *
from preprocessing import *
from ui import get_theme, style_fig, log


def processing_tab(df):
    t, pal, _ = get_theme()

    st.markdown('<div class="sec-header">🔄 Data Processing Pipeline</div>', unsafe_allow_html=True)

    proc_subtab = st.tabs([
        "📊 Overview", "🕳️ Missing Values", "🔄 Duplicates",
        "⚠️ Outliers", "📝 Inconsistent Data",
        "🏷️ Encoding", "📏 Scaling", "🎯 Feature Selection", "✂️ Train/Test Split",
    ])

    # ── Overview ──
    with proc_subtab[0]:
        st.markdown("#### 📊 Dataset Summary")
        rows, cols_count = get_shape(df)
        numeric_cols, categorical_cols = detect_columns(df)
        missing_df   = check_missing_values(df).toPandas()
        total_missing = missing_df['missing_count'].sum() if not missing_df.empty else 0

        c1, c2, c3, c4, c5 = st.columns(5)
        c1.metric("Rows",          f"{rows:,}")
        c2.metric("Columns",       f"{cols_count:,}")
        c3.metric("Numeric",       f"{len(numeric_cols)}")
        c4.metric("Categorical",   f"{len(categorical_cols)}")
        c5.metric("Missing Total", f"{int(total_missing):,}")

        st.markdown("#### Column Information")
        col_info = [{"Column": cn, "Type": ct} for cn, ct in df.dtypes]
        st.dataframe(pd.DataFrame(col_info), use_container_width=True)

    # ── Missing Values ──
    with proc_subtab[1]:
        st.markdown("#### 🕳️ Missing Values Analysis")
        missing_df   = check_missing_values(df).toPandas()
        missing_cols_df = missing_df[missing_df['missing_count'] > 0]

        if not missing_cols_df.empty:
            st.dataframe(missing_cols_df, use_container_width=True)

            if st.button("📊 Show Missing Values Chart", key="show_missing_chart"):
                fig, ax = plt.subplots(figsize=(8, 4))
                mc_sorted = missing_cols_df.sort_values('missing_count', ascending=False)
                ax.barh(mc_sorted['column_name'], mc_sorted['missing_count'], color=t["danger"])
                ax.set_title('Missing Values per Column', color=t["text"])
                ax.set_xlabel('Count', color=t["muted"])
                st.pyplot(style_fig(fig)); plt.close()

            st.markdown("#### 🛠️ Handle Missing Values")
            col1, col2 = st.columns(2)
            with col1:
                mv_col      = st.selectbox("Column to treat", missing_cols_df['column_name'].tolist(), key="mv_col_select")
                mv_strategy = st.selectbox("Strategy",
                    ["Drop rows", "Fill with Mean", "Fill with Median", "Fill with Mode", "Fill with Constant"],
                    key="mv_strategy_select")
            with col2:
                mv_const = ""
                if mv_strategy == "Fill with Constant":
                    mv_const = st.text_input("Constant value", "0", key="mv_const")

            if st.button("✅ Apply Treatment", key="apply_missing_btn"):
                try:
                    with st.spinner("Processing..."):
                        if mv_strategy == "Drop rows":
                            new_df = drop_rows_missing_cols(df, [mv_col])
                            log(f"Dropped rows with missing in '{mv_col}'")
                        elif mv_strategy == "Fill with Mean":
                            new_df = impute_mean(df, [mv_col])
                            log(f"Filled '{mv_col}' with mean")
                        elif mv_strategy == "Fill with Median":
                            new_df = impute_median(df, [mv_col])
                            log(f"Filled '{mv_col}' with median")
                        elif mv_strategy == "Fill with Mode":
                            new_df = impute_mode(df, [mv_col])
                            log(f"Filled '{mv_col}' with mode")
                        else:
                            new_df = df.fillna({mv_col: mv_const})
                            log(f"Filled '{mv_col}' with constant '{mv_const}'")

                        st.session_state.spark_df = new_df
                        st.session_state.train_df = None
                        st.session_state.test_df  = None
                        st.success(f"✅ Applied '{mv_strategy}' on '{mv_col}'")
                        st.rerun()
                except Exception as e:
                    st.error(f"❌ Error: {e}")
                    log(f"ERROR missing treatment: {e}")

            if st.checkbox("Show data preview", key="show_preview"):
                st.dataframe(st.session_state.spark_df.limit(20).toPandas(), use_container_width=True)
        else:
            st.success("✅ No missing values found!")

    # ── Duplicates ──
    with proc_subtab[2]:
        st.markdown("#### 🔄 Duplicates Analysis")
        duplicates, total_rows, unique_rows = count_duplicates_spark(df)
        c1, c2, c3 = st.columns(3)
        c1.metric("Total Rows",     f"{total_rows:,}")
        c2.metric("Unique Rows",    f"{unique_rows:,}")
        c3.metric("Duplicate Rows", f"{duplicates:,}",
                  delta=f"{(duplicates/total_rows)*100:.2f}%" if total_rows > 0 else None)

        if duplicates > 0:
            if st.button("🗑️ Remove Duplicates", key="remove_dupes"):
                new_df = remove_duplicates_spark(df)
                st.session_state.spark_df = new_df
                log(f"Removed {duplicates} duplicate rows")
                st.success(f"✅ Removed {duplicates} duplicates. New size: {new_df.count():,} rows")
                st.rerun()
            if st.button("🔍 Show Duplicate Groups", key="show_dupes"):
                dup_groups = df.groupBy(df.columns).count().filter(col("count") > 1)
                st.write(f"Duplicate groups: {dup_groups.count()}")
                st.dataframe(dup_groups.limit(50).toPandas(), use_container_width=True)
        else:
            st.info("ℹ️ No duplicate rows found.")

    # ── Outliers ──
    with proc_subtab[3]:
        st.markdown("#### ⚠️ Outlier Detection & Treatment")
        numeric_cols, _ = detect_columns(df)

        if numeric_cols:
            outlier_col      = st.selectbox("Column", numeric_cols, key="outlier_col_select")
            detection_method = st.selectbox("Detection Method", ["IQR", "Z-Score", "Percentile"], key="detection_method")

            threshold = 3
            lower_pct, upper_pct = 0.01, 0.99
            if detection_method == "Z-Score":
                threshold = st.slider("Z-Score Threshold", 1.0, 5.0, 3.0, 0.5, key="zscore_threshold")
            elif detection_method == "Percentile":
                col1, col2 = st.columns(2)
                with col1:
                    lower_pct = st.slider("Lower Percentile", 0.01, 0.10, 0.01, 0.01, key="lower_pct")
                with col2:
                    upper_pct = st.slider("Upper Percentile", 0.90, 0.99, 0.99, 0.01, key="upper_pct")

            if st.button("🔍 Detect Outliers", key="detect_btn"):
                try:
                    before_count = df.count()
                    if detection_method == "IQR":
                        outliers, out_count, lower, upper = iqr_outlier_detection(df, outlier_col)
                        st.markdown(f"""| Metric | Value |
|--------|-------|
| **Lower Bound (Q1 - 1.5×IQR)** | `{lower:.4f}` |
| **Upper Bound (Q3 + 1.5×IQR)** | `{upper:.4f}` |
| **Outliers Detected** | `{out_count:,}` |
| **Percentage** | `{(out_count/before_count)*100:.2f}%` |""")
                    elif detection_method == "Z-Score":
                        outliers, out_count, mean_val, std_val = detect_outliers_zscore(df, outlier_col, threshold)
                        st.markdown(f"""| Metric | Value |
|--------|-------|
| **Mean** | `{mean_val:.4f}` |
| **Std Dev** | `{std_val:.4f}` |
| **Threshold** | `{threshold}` |
| **Outliers Detected** | `{out_count:,}` |
| **Percentage** | `{(out_count/before_count)*100:.2f}%` |""")
                    else:
                        outliers, out_count, lower, upper = detect_outliers_percentile(df, outlier_col, lower_pct, upper_pct)
                        st.markdown(f"""| Metric | Value |
|--------|-------|
| **Lower ({lower_pct*100:.0f}%)** | `{lower:.4f}` |
| **Upper ({upper_pct*100:.0f}%)** | `{upper:.4f}` |
| **Outliers Detected** | `{out_count:,}` |
| **Percentage** | `{(out_count/before_count)*100:.2f}%` |""")

                    if out_count > 0:
                        st.warning(f"⚠️ Found {out_count:,} outliers")
                        st.dataframe(outliers.limit(50).toPandas(), use_container_width=True)
                    else:
                        st.success("✅ No outliers detected!")
                except Exception as e:
                    st.error(f"Error: {e}")

            st.markdown("#### 🛠️ Outlier Treatment")
            treatment_method = st.selectbox("Treatment",
                ["Remove", "Replace with Mean", "Replace with Median", "Log Transform"],
                key="treatment_method")

            if st.button("Apply Treatment", key="apply_treatment"):
                try:
                    if detection_method == "IQR":
                        _, _, lower, upper = iqr_outlier_detection(df, outlier_col)
                    elif detection_method == "Z-Score":
                        _, _, mean_val, std_val = detect_outliers_zscore(df, outlier_col, threshold)
                        lower = mean_val - threshold * std_val
                        upper = mean_val + threshold * std_val
                    else:
                        _, _, lower, upper = detect_outliers_percentile(df, outlier_col, lower_pct, upper_pct)

                    condition = (col(outlier_col) < lower) | (col(outlier_col) > upper)

                    if treatment_method == "Remove":
                        new_df = df.filter(~condition)
                        log(f"Removed outliers from '{outlier_col}'")
                        st.success(f"✅ Removed outliers. New size: {new_df.count():,}")
                    elif treatment_method == "Replace with Mean":
                        mean_val = df.select(mean(outlier_col)).collect()[0][0]
                        new_df = df.withColumn(outlier_col, F.when(condition, mean_val).otherwise(col(outlier_col)))
                        log(f"Replaced outliers with mean in '{outlier_col}'")
                        st.success(f"✅ Replaced with mean: {mean_val:.4f}")
                    elif treatment_method == "Replace with Median":
                        median_val = df.approxQuantile(outlier_col, [0.5], 0.01)[0]
                        new_df = df.withColumn(outlier_col, F.when(condition, median_val).otherwise(col(outlier_col)))
                        log(f"Replaced outliers with median in '{outlier_col}'")
                        st.success(f"✅ Replaced with median: {median_val:.4f}")
                    else:
                        new_df = df.withColumn(outlier_col, log1p(col(outlier_col)))
                        log(f"Applied log transform to '{outlier_col}'")
                        st.success("✅ Log transformation applied.")

                    st.session_state.spark_df = new_df
                    st.rerun()
                except Exception as e:
                    st.error(f"Error: {e}")
        else:
            st.info("ℹ️ No numeric columns available.")

    # ── Inconsistent Data ──
    with proc_subtab[4]:
        st.markdown("#### 📝 Inconsistent Data Handling")

        st.markdown("##### 🔍 Unique Values")
        col_to_check = st.selectbox("Column", df.columns, key="unique_col")
        if st.button("Show Unique Values", key="show_unique"):
            unique_vals = check_unique_values(df, col_to_check)
            st.write(f"Unique count: {unique_vals.count()}")
            st.dataframe(unique_vals.limit(100).toPandas(), use_container_width=True)

        st.markdown("##### 🧹 Clean Text Columns")
        text_cols = [c for c, tp in df.dtypes if tp == "string"]
        if text_cols:
            cols_to_clean = st.multiselect("Select columns", text_cols, key="clean_cols")
            if cols_to_clean and st.button("Clean Text Columns", key="clean_text"):
                new_df = clean_text_columns(df, cols_to_clean)
                st.session_state.spark_df = new_df
                log(f"Cleaned text columns: {cols_to_clean}")
                st.success(f"✅ Cleaned {len(cols_to_clean)} text columns")
                st.rerun()
        else:
            st.info("No text columns found.")

        st.markdown("##### 📅 Standardize Date Columns")
        date_cols = [c for c, tp in df.dtypes if tp == "string" and "date" in c.lower()]
        if date_cols:
            date_col    = st.selectbox("Date column", date_cols, key="date_col")
            date_format = st.text_input("Date format", "yyyy-MM-dd", key="date_format")
            if st.button("Standardize Date", key="standardize_date"):
                new_df = standardize_date(df, date_col, date_format)
                st.session_state.spark_df = new_df
                log(f"Standardized date column: {date_col}")
                st.success(f"✅ Standardized '{date_col}'")
                st.rerun()

        st.markdown("##### 🔢 Clean Numeric-Text Columns")
        if text_cols:
            num_text_col = st.selectbox("Column with numeric text", text_cols, key="num_text_col")
            if st.button("Clean Numeric Text", key="clean_numeric"):
                new_df = clean_numeric_text(df, num_text_col)
                st.session_state.spark_df = new_df
                log(f"Cleaned numeric text in: {num_text_col}")
                st.success(f"✅ Cleaned numeric text in '{num_text_col}'")
                st.rerun()

    # ── Encoding ──
    with proc_subtab[5]:
        st.markdown("#### 🏷️ Categorical Encoding")
        st.info("⚠️ Split your data first in 'Train/Test Split' before encoding.")

        train_exists = st.session_state.train_df is not None
        test_exists  = st.session_state.test_df  is not None

        if train_exists and test_exists:
            tr = st.session_state.train_df
            te = st.session_state.test_df
            _, categorical_cols = detect_columns(tr)

            if categorical_cols:
                enc_col    = st.selectbox("Column to encode", categorical_cols, key="enc_col")
                enc_method = st.selectbox("Method", ["Label Encoding", "One-Hot Encoding", "Frequency Encoding"], key="enc_method")
                if st.button("Apply Encoding", key="apply_encoding"):
                    try:
                        if enc_method == "Label Encoding":
                            tr_enc, te_enc = apply_label_encoding_spark(tr, te, enc_col)
                        elif enc_method == "One-Hot Encoding":
                            tr_enc, te_enc = apply_onehot_encoding_spark(tr, te, enc_col)
                        else:
                            tr_enc, te_enc = frequency_encoding(tr, te, [enc_col])
                        st.session_state.train_df = tr_enc
                        st.session_state.test_df  = te_enc
                        log(f"Applied {enc_method} on '{enc_col}'")
                        st.success(f"✅ Applied {enc_method} on '{enc_col}'")
                    except Exception as e:
                        st.error(f"Error: {e}")
            else:
                st.info("No categorical columns found.")
        else:
            st.warning("⚠️ Split your data first.")

    # ── Scaling ──
    with proc_subtab[6]:
        st.markdown("#### 📏 Feature Scaling")
        train_exists = st.session_state.train_df is not None

        if train_exists:
            tr = st.session_state.train_df
            te = st.session_state.test_df
            numeric_cols, _ = detect_columns(tr)

            if numeric_cols:
                scaler_method = st.selectbox("Scaling Method",
                    ["Standard Scaler (Z-score)", "MinMax Scaler"], key="scaler_method")
                if st.button("Apply Scaling", key="apply_scaling"):
                    try:
                        if scaler_method == "Standard Scaler (Z-score)":
                            tr_s, te_s = apply_standard_scaler_spark(tr, te)
                        else:
                            tr_s, te_s = apply_minmax_scaler_spark(tr, te)
                        st.session_state.train_df = tr_s
                        st.session_state.test_df  = te_s
                        log(f"Applied {scaler_method}")
                        st.success(f"✅ Applied {scaler_method}")
                    except Exception as e:
                        st.error(f"Error: {e}")
            else:
                st.info("No numeric columns found.")
        else:
            st.warning("⚠️ Split your data first.")

    # ── Feature Selection ──
    with proc_subtab[7]:
        st.markdown("#### 🎯 Feature Selection")
        train_exists = st.session_state.train_df is not None

        if train_exists:
            tr = st.session_state.train_df
            numeric_cols, _ = detect_columns(tr)

            if numeric_cols:
                fs_method = st.selectbox("Method", ["Variance Threshold", "Chi-Square"], key="fs_method")
                if fs_method == "Variance Threshold":
                    var_thresh = st.slider("Variance Threshold", 0.0, 0.1, 0.01, 0.001, key="var_threshold")
                    if st.button("Select by Variance", key="select_variance"):
                        selected = select_features_variance_spark(tr, var_thresh)
                        st.success(f"✅ Selected {len(selected)} features (variance > {var_thresh})")
                        st.write("**Selected Features:**", selected)
                        log(f"Variance selection: {len(selected)} features")
                else:
                    label_col = st.selectbox("Target column", tr.columns, key="label_col_chisq")
                    k_feat    = st.slider("Top k features", 1, min(20, len(numeric_cols)), 5, key="k_features")
                    if st.button("Select by Chi-Square", key="select_chisq"):
                        selected = select_features_chisq_spark(tr, label_col, k_feat)
                        st.success(f"✅ Selected {len(selected)} top features")
                        st.write("**Selected Features:**", selected)
                        log(f"Chi-Square selection: {len(selected)} features")
            else:
                st.info("No numeric columns found.")
        else:
            st.warning("⚠️ Split your data first.")

    # ── Train/Test Split ──
    with proc_subtab[8]:
        st.markdown("#### ✂️ Train/Test Split")
        test_size   = st.slider("Test Size Ratio", 0.1, 0.5, 0.2, 0.05, key="test_size_split")
        random_seed = st.number_input("Random Seed", value=42, key="random_seed")

        if st.button("Split Data", key="split_data"):
            train, test = split_data_spark(df, test_size, random_seed)
            st.session_state.train_df = train
            st.session_state.test_df  = test
            train_n = train.count()
            test_n  = test.count()
            st.success(f"✅ Train: {train_n:,} rows | Test: {test_n:,} rows")
            log(f"Split data — Train: {train_n}, Test: {test_n}")

            st.markdown("#### Train Set Preview")
            st.dataframe(train.limit(10).toPandas(), use_container_width=True)
            st.markdown("#### Test Set Preview")
            st.dataframe(test.limit(10).toPandas(), use_container_width=True)

        if st.session_state.train_df is not None:
            st.info(
                f"Current split — Train: {st.session_state.train_df.count():,} rows | "
                f"Test: {st.session_state.test_df.count():,} rows"
            )

Writing processing_tab.py


### modeling_tab file

In [118]:
%%writefile modeling_tab.py
import streamlit as st

from ui import get_theme, themed_bar_chart, result_card_html, model_banner_html, log


def modeling_tab(df):
    import pandas as pd
    import matplotlib.pyplot as plt
    import matplotlib; matplotlib.use("Agg")
    import numpy as np
    from pyspark.ml.feature import VectorAssembler, StringIndexer
    from pyspark.sql import functions as F

    from models import (
        detect_task_type,
        get_available_classification_models,
        train_classification_model,
        evaluate_classification_model,
        select_best_model,
        XGBOOST_AVAILABLE,
        get_available_regression_models,
        train_regression_model,
        evaluate_regression_model,
        select_best_regression_model,
    )

    t, pal, _ = get_theme()

    st.markdown('<div class="sec-header">🤖 Machine Learning Workspace</div>', unsafe_allow_html=True)

    if st.session_state.train_df is None or st.session_state.test_df is None:
        st.warning("⚠️ Please split your data first in **Processing → Train/Test Split**.")
        return

    train_df = st.session_state.train_df
    test_df  = st.session_state.test_df

    cls_tab, reg_tab = st.tabs(["🏷️  Classification", "📈  Regression"])

    # ── Shared helpers ──────────────────────────────────────────────────────
    def prepare_data(tr, te, feat_cols, tgt, do_index):
        assembler = VectorAssembler(inputCols=feat_cols, outputCol="features", handleInvalid="skip")
        tr = assembler.transform(tr)
        te = assembler.transform(te)
        if do_index:
            indexer   = StringIndexer(inputCol=tgt, outputCol=tgt + "_idx", handleInvalid="keep")
            idx_model = indexer.fit(tr)
            tr = idx_model.transform(tr)
            te = idx_model.transform(te)
            return tr, te, tgt + "_idx"
        return tr, te, tgt

    def get_candidate_features(df_ref, exclude_col):
        nt = ("int", "bigint", "double", "float", "decimal", "long", "short", "tinyint")
        numeric  = [c for c, tp in df_ref.dtypes if any(x in tp.lower() for x in nt) and c != exclude_col]
        encoded  = [c for c in df_ref.columns if (c.endswith("_indexed") or c.endswith("_ohe")) and c != exclude_col]
        return list(dict.fromkeys(numeric + encoded))

    def highlight_max(s):
        is_max = s == s.max()
        return [f"background-color:{t['accent']}22;color:{t['accent']};font-weight:700" if v else "" for v in is_max]

    def highlight_reg(df_s):
        styles = pd.DataFrame("", index=df_s.index, columns=df_s.columns)
        for cn in df_s.columns:
            idx = df_s[cn].idxmax() if cn == "R2" else df_s[cn].idxmin()
            styles.loc[idx, cn] = f"background-color:{t['accent']}22;color:{t['accent']};font-weight:700"
        return styles

    # ── Prediction input widget ──────────────────────────────────────────────
    def prediction_inputs(feat_cols, key_prefix):
        input_vals = {}
        ncols      = 3
        groups     = [feat_cols[i:i+ncols] for i in range(0, len(feat_cols), ncols)]
        for group in groups:
            cols_ui = st.columns(len(group))
            for ci, fc in enumerate(group):
                dtype = dict(train_df.dtypes).get(fc, "double")
                if any(x in dtype for x in ("int", "long", "bigint", "short", "tinyint")):
                    input_vals[fc] = cols_ui[ci].number_input(fc, value=0, step=1, key=f"{key_prefix}_{fc}")
                else:
                    input_vals[fc] = cols_ui[ci].number_input(fc, value=0.0, step=0.01, format="%.4f", key=f"{key_prefix}_{fc}")
        return input_vals

    # ==========================================================================
    # CLASSIFICATION
    # ==========================================================================
    with cls_tab:
        st.markdown("### 🎯 Step 1 — Target Column")
        cls_target = st.selectbox("Target column", train_df.columns, key="cls_target")

        with st.spinner("Detecting task type..."):
            task_type = detect_task_type(train_df, cls_target)
            n_classes = train_df.select(cls_target).distinct().count()

        task_color = t["accent"] if task_type == "binary" else t["secondary"]
        st.markdown(
            f"**Task:** <span style='color:{task_color};font-family:Space Mono,monospace;font-weight:700;'>"
            f"{task_type.upper()} ({n_classes} classes)</span>",
            unsafe_allow_html=True,
        )
        if task_type == "multiclass":
            classes = [r[0] for r in train_df.select(cls_target).distinct().collect()]
            st.caption(f"Classes: {classes}")

        st.markdown("---")
        st.markdown("### ⚙️ Step 2 — Feature Columns")
        cls_candidates = get_candidate_features(train_df, cls_target)
        if not cls_candidates:
            st.error("❌ No usable feature columns found. Preprocess your data first.")
            st.stop()

        cls_features = st.multiselect("Features", cls_candidates, default=cls_candidates, key="cls_features")
        if not cls_features:
            st.warning("Select at least one feature.")
            st.stop()

        cls_needs_index = dict(train_df.dtypes).get(cls_target, "") == "string"
        if cls_needs_index:
            st.info(f"ℹ️ Target `{cls_target}` is string — will be auto-indexed.")

        st.markdown("---")
        st.markdown("### 🧪 Step 3 — Choose Models")
        available_cls = get_available_classification_models()
        if not XGBOOST_AVAILABLE:
            st.info("ℹ️ XGBoost not installed (`pip install xgboost[spark]`).")
        if task_type == "multiclass":
            st.caption("⚠️ GBT supports binary only natively. SVM uses OneVsRest automatically.")

        selected_cls = st.multiselect("Models to train", available_cls, default=available_cls[:2], key="selected_cls_models")
        st.markdown("---")

        if st.button("🚀 Train & Evaluate", key="cls_train_btn"):
            if not selected_cls:
                st.warning("Select at least one model."); st.stop()

            with st.spinner("Assembling feature vector..."):
                try:
                    tr_r, te_r, cls_label = prepare_data(train_df, test_df, cls_features, cls_target, cls_needs_index)
                except Exception as e:
                    st.error(f"❌ Assembly failed: {e}"); st.stop()

            st.success(f"✅ Feature vector: {len(cls_features)} columns | Label: `{cls_label}`")

            cls_results = {}; cls_models_trained = {}; cls_errors = {}
            prog = st.progress(0); status = st.empty()

            for i, mname in enumerate(selected_cls):
                status.markdown(f"⏳ Training **{mname}**...")
                try:
                    m = train_classification_model(mname, tr_r, cls_label, task_type)
                    cls_results[mname] = evaluate_classification_model(m, te_r, cls_label, task_type)
                    cls_models_trained[mname] = m
                except Exception as e:
                    cls_errors[mname] = str(e)
                prog.progress(int((i + 1) / len(selected_cls) * 100))

            status.empty(); prog.empty()
            for name, err in cls_errors.items():
                st.error(f"❌ **{name}** failed: {err}")
            if not cls_results:
                st.error("All models failed."); st.stop()

            st.session_state.cls_models      = cls_models_trained
            st.session_state.cls_results     = cls_results
            st.session_state.cls_task_type   = task_type
            st.session_state.cls_target      = cls_target
            st.session_state.cls_features    = cls_features
            st.session_state.cls_needs_index = cls_needs_index
            st.session_state.cls_label_col   = cls_label

            st.markdown("## 📊 Evaluation Results")
            rdf = pd.DataFrame(cls_results).T.round(4)
            st.dataframe(rdf.style.apply(highlight_max, axis=0), use_container_width=True)

            best_name, best_metrics = select_best_model(cls_results, "F1")
            st.markdown(
                model_banner_html(
                    "🏆 BEST MODEL (by F1)", best_name,
                    f"{t['primary']}14", f"{t['primary']}55", t["primary"],
                ),
                unsafe_allow_html=True,
            )

            st.markdown("### 🏆 Best Model Metrics")
            mc = st.columns(len(best_metrics))
            for ci, (k, v) in enumerate(best_metrics.items()):
                mc[ci].metric(k, f"{v:.4f}")

            st.markdown("### 📈 Model Comparison")
            mplot = [m for m in ["Accuracy", "F1", "Precision", "Recall", "AUC-ROC", "AUC-PR"] if m in rdf.columns]
            fig = themed_bar_chart(rdf, mplot, "Classification Comparison")
            st.pyplot(fig); plt.close()

            st.markdown("### 🔍 Per-Model Detail")
            for mname, metrics in cls_results.items():
                is_best = mname == best_name
                with st.expander(f"{'🥇' if is_best else '📋'} {mname}{'  ← Best' if is_best else ''}", expanded=is_best):
                    dc = st.columns(len(metrics))
                    for ci, (k, v) in enumerate(metrics.items()):
                        dc[ci].metric(k, f"{v:.4f}")

            log(f"[CLS] {list(cls_results.keys())} | Best: {best_name} | F1={best_metrics['F1']:.4f}")

        # ── Classification Prediction ────────────────────────────────────────
        if st.session_state.get("cls_models") and st.session_state.get("cls_features"):
            st.markdown("---")
            st.markdown("## 🔮 Predict — Classification")
            st.caption("Enter feature values and click Predict.")

            pred_model_name = st.selectbox("Model", list(st.session_state.cls_models.keys()), key="pred_cls_model_name")
            input_vals = prediction_inputs(st.session_state.cls_features, "cls_inp")

            if st.button("🎯 Predict", key="cls_predict_btn"):
                try:
                    feat_cols   = st.session_state.cls_features
                    spark_local = train_df.sparkSession
                    input_df    = spark_local.createDataFrame([tuple(input_vals[fc] for fc in feat_cols)], feat_cols)
                    assembler   = VectorAssembler(inputCols=feat_cols, outputCol="features", handleInvalid="keep")
                    assembled   = assembler.transform(input_df)

                    pred_result = st.session_state.cls_models[pred_model_name].transform(assembled)
                    prediction  = pred_result.select("prediction").collect()[0][0]

                    cls_target_col = st.session_state.get("cls_target", "target")
                    unique_labels  = [r[0] for r in train_df.select(cls_target_col).distinct().orderBy(cls_target_col).collect()]
                    pred_int  = int(prediction)
                    label_str = str(unique_labels[pred_int]) if pred_int < len(unique_labels) else str(pred_int)

                    st.markdown(
                        result_card_html(
                            "PREDICTION RESULT", label_str,
                            f"Model: {pred_model_name}",
                            f"{t['accent']}10", f"{t['accent']}50", t["accent"],
                        ),
                        unsafe_allow_html=True,
                    )

                    try:
                        prob      = pred_result.select("probability").collect()[0][0]
                        prob_list = prob.toArray().tolist()
                        prob_df   = pd.DataFrame({
                            "Class": [str(l) for l in unique_labels[:len(prob_list)]],
                            "Probability": [round(p, 4) for p in prob_list],
                        })
                        st.markdown("**Class Probabilities:**")
                        st.dataframe(prob_df, use_container_width=True)
                    except Exception:
                        pass
                except Exception as e:
                    st.error(f"❌ Prediction failed: {e}")

    # ==========================================================================
    # REGRESSION
    # ==========================================================================
    with reg_tab:
        st.markdown("### 🎯 Step 1 — Target Column")
        reg_target = st.selectbox("Target column (numeric)", train_df.columns, key="reg_target")

        reg_dtype = dict(train_df.dtypes).get(reg_target, "string")
        nt_check  = ("int", "bigint", "double", "float", "decimal", "long", "short", "tinyint")
        if not any(x in reg_dtype.lower() for x in nt_check):
            st.warning(f"⚠️ `{reg_target}` is type `{reg_dtype}`. Regression requires a numeric target.")

        try:
            stats = train_df.select(
                F.mean(reg_target).alias("mean"), F.stddev(reg_target).alias("std"),
                F.min(reg_target).alias("min"),   F.max(reg_target).alias("max"),
            ).collect()[0]
            c1, c2, c3, c4 = st.columns(4)
            c1.metric("Mean", f"{stats['mean']:.4f}" if stats['mean'] else "N/A")
            c2.metric("Std",  f"{stats['std']:.4f}"  if stats['std']  else "N/A")
            c3.metric("Min",  f"{stats['min']:.4f}"  if stats['min']  else "N/A")
            c4.metric("Max",  f"{stats['max']:.4f}"  if stats['max']  else "N/A")
        except Exception:
            pass

        st.markdown("---")
        st.markdown("### ⚙️ Step 2 — Feature Columns")
        reg_candidates = get_candidate_features(train_df, reg_target)
        if not reg_candidates:
            st.error("❌ No usable feature columns found."); st.stop()

        reg_features = st.multiselect("Features", reg_candidates, default=reg_candidates, key="reg_features")
        if not reg_features:
            st.warning("Select at least one feature."); st.stop()

        st.markdown("---")
        st.markdown("### 🧪 Step 3 — Choose Models")
        available_reg = get_available_regression_models()
        st.caption("ℹ️ Isotonic Regression works best with a single monotonic feature.")

        selected_reg = st.multiselect("Models to train", available_reg, default=available_reg[:2], key="selected_reg_models")
        st.markdown("---")

        if st.button("🚀 Train & Evaluate", key="reg_train_btn"):
            if not selected_reg:
                st.warning("Select at least one model."); st.stop()

            with st.spinner("Assembling feature vector..."):
                try:
                    tr_r, te_r, reg_label = prepare_data(train_df, test_df, reg_features, reg_target, do_index=False)
                except Exception as e:
                    st.error(f"❌ Assembly failed: {e}"); st.stop()

            st.success(f"✅ Feature vector: {len(reg_features)} columns | Label: `{reg_label}`")

            reg_results = {}; reg_models_trained = {}; reg_errors = {}
            prog = st.progress(0); status = st.empty()

            for i, mname in enumerate(selected_reg):
                status.markdown(f"⏳ Training **{mname}**...")
                try:
                    m = train_regression_model(mname, tr_r, reg_label)
                    reg_results[mname] = evaluate_regression_model(m, te_r, reg_label)
                    reg_models_trained[mname] = m
                except Exception as e:
                    reg_errors[mname] = str(e)
                prog.progress(int((i + 1) / len(selected_reg) * 100))

            status.empty(); prog.empty()
            for name, err in reg_errors.items():
                st.error(f"❌ **{name}** failed: {err}")
            if not reg_results:
                st.error("All models failed."); st.stop()

            st.session_state.reg_models   = reg_models_trained
            st.session_state.reg_results  = reg_results
            st.session_state.reg_target   = reg_target
            st.session_state.reg_features = reg_features

            st.markdown("## 📊 Evaluation Results")
            rdf_r = pd.DataFrame(reg_results).T.round(4)
            st.dataframe(rdf_r.style.apply(highlight_reg, axis=None), use_container_width=True)

            best_reg_name, best_reg_metrics = select_best_regression_model(reg_results, "R2")
            st.markdown(
                model_banner_html(
                    "🏆 BEST MODEL (by R²)", best_reg_name,
                    f"{t['secondary']}10", f"{t['secondary']}55", t["secondary"],
                ),
                unsafe_allow_html=True,
            )

            st.markdown("### 🏆 Best Model Metrics")
            cr = st.columns(len(best_reg_metrics))
            for ci, (k, v) in enumerate(best_reg_metrics.items()):
                cr[ci].metric(k, f"{v:.4f}")

            st.markdown("### 📈 Model Comparison")
            fig_r = themed_bar_chart(rdf_r, list(rdf_r.columns), "Regression Comparison")
            st.pyplot(fig_r); plt.close()

            st.markdown("### 🔍 Per-Model Detail")
            for mname, metrics in reg_results.items():
                is_best = mname == best_reg_name
                with st.expander(f"{'🥇' if is_best else '📋'} {mname}{'  ← Best' if is_best else ''}", expanded=is_best):
                    dc = st.columns(len(metrics))
                    for ci, (k, v) in enumerate(metrics.items()):
                        dc[ci].metric(k, f"{v:.4f}")

            log(f"[REG] {list(reg_results.keys())} | Best: {best_reg_name} | R2={best_reg_metrics['R2']:.4f}")

        # ── Regression Prediction ─────────────────────────────────────────────
        if st.session_state.get("reg_models") and st.session_state.get("reg_features"):
            st.markdown("---")
            st.markdown("## 🔮 Predict — Regression")
            st.caption("Enter feature values and click Predict.")

            pred_reg_name = st.selectbox("Model", list(st.session_state.reg_models.keys()), key="pred_reg_model_name")
            reg_input_vals = prediction_inputs(st.session_state.reg_features, "reg_inp")

            if st.button("📐 Predict Value", key="reg_predict_btn"):
                try:
                    feat_cols   = st.session_state.reg_features
                    spark_local = train_df.sparkSession
                    input_df    = spark_local.createDataFrame([tuple(reg_input_vals[fc] for fc in feat_cols)], feat_cols)
                    assembler   = VectorAssembler(inputCols=feat_cols, outputCol="features", handleInvalid="keep")
                    assembled   = assembler.transform(input_df)

                    pred_result = st.session_state.reg_models[pred_reg_name].transform(assembled)
                    prediction  = pred_result.select("prediction").collect()[0][0]

                    st.markdown(
                        result_card_html(
                            "PREDICTED VALUE", f"{prediction:.4f}",
                            f"Model: {pred_reg_name} | Target: {st.session_state.reg_target}",
                            f"{t['secondary']}10", f"{t['secondary']}50", t["secondary"],
                        ),
                        unsafe_allow_html=True,
                    )
                except Exception as e:
                    st.error(f"❌ Prediction failed: {e}")

Writing modeling_tab.py


### app file

In [119]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
import warnings
import io
import os
import pickle
import traceback
from datetime import datetime
from EDA import *
from preprocessing import *

# PySpark imports
import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import col, sum, when, mean, trim, lower, to_date, regexp_replace, log1p, variance
from pyspark.sql.types import NumericType
import missingno as msno

# PySpark ML imports
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler, MinMaxScaler, ChiSqSelector
from pyspark.sql.functions import min as spark_min

warnings.filterwarnings('ignore')

from ui import get_theme, inject_css, log
from data_loader import get_spark_session, load_csv_spark, load_kaggle_spark, spark
from visualization_tab import visualization_tab
from EDA_tab import eda_tab
from processing_tab import processing_tab
from modeling_tab import modeling_tab

# ==============================================================================
# PAGE CONFIG
# ==============================================================================
st.set_page_config(
    page_title="DataSci Studio",
    page_icon="🔬",
    layout="wide",
    initial_sidebar_state="expanded",
)

# ==============================================================================
# SESSION STATE INIT
# ==============================================================================
defaults = {
    "spark_df": None,
    "spark_df_original": None,
    "logs": [],
    "predictions": None,
    "model": None,
    "train_df": None,
    "test_df": None,
    "theme_mode": "dark",
}
for k, v in defaults.items():
    if k not in st.session_state:
        st.session_state[k] = v

# Resolve theme early so CSS can be injected
theme, palette, mode = get_theme()
inject_css(theme, mode)


# ==============================================================================
# HELPERS
# ==============================================================================
def get_col_type(df, col_name: str) -> str:
    col_type = dict(df.dtypes).get(col_name, "string")
    numeric_types = ['int', 'bigint', 'double', 'float', 'decimal', 'long', 'short', 'tinyint']
    if any(nt in col_type.lower() for nt in numeric_types):
        return "numeric"
    return "categorical"


# ==============================================================================
# SIDEBAR
# ==============================================================================
with st.sidebar:
    t, pal, mode = get_theme()

    st.markdown(
        '<div class="logo-area">Data<span>Sci</span> Studio</div>'
        '<div class="logo-tagline">End-to-end ML workspace · PySpark</div>',
        unsafe_allow_html=True,
    )
    st.divider()

    # Theme toggle
    st.markdown('<div class="theme-toggle-label">🎨 Theme</div>', unsafe_allow_html=True)
    theme_choice = st.radio(
        "theme_radio",
        ["🌙 Dark", "☀️ Light"],
        index=0 if mode == "dark" else 1,
        horizontal=True,
        label_visibility="collapsed",
        key="theme_radio",
    )
    new_mode = "dark" if "Dark" in theme_choice else "light"
    if new_mode != st.session_state.theme_mode:
        st.session_state.theme_mode = new_mode
        st.rerun()

    st.divider()

    # Data source
    st.markdown("### 📂 Data Source")
    source = st.radio("Input method", ["Upload CSV", "Kaggle Path"],
                      horizontal=True, label_visibility='collapsed')

    if source == "Upload CSV":
        uploaded    = st.file_uploader("Choose CSV file", type=['csv'], label_visibility='collapsed')
        sample_rows = st.number_input("Sample rows (0 = all)", min_value=0, value=100000, step=10000)

        if uploaded and st.button("Load Dataset"):
            with st.spinner("Loading with PySpark..."):
                try:
                    import tempfile
                    with tempfile.NamedTemporaryFile(delete=False, suffix='.csv') as tmp:
                        tmp.write(uploaded.getvalue())
                        tmp_path = tmp.name
                    nrows = sample_rows if sample_rows > 0 else None
                    df = load_csv_spark(tmp_path, nrows=nrows)
                    df = df.cache(); df.count()
                    st.session_state.spark_df          = df
                    st.session_state.spark_df_original = df
                    st.session_state.train_df          = None
                    st.session_state.test_df           = None
                    r, c = get_shape(df)
                    log(f"Loaded CSV: {r:,} rows × {c} cols")
                    st.success(f"✅ Loaded {r:,} rows")
                    os.unlink(tmp_path)
                except Exception as e:
                    st.error(f"Error: {e}"); log(f"ERROR: {e}")
    else:
        kg_path = st.text_input("Dataset ID", placeholder="username/dataset-name")
        kg_file = st.text_input("Filename", placeholder="data.csv")
        kg_rows = st.number_input("Max rows (0 = all)", min_value=0, value=100000, step=10000)

        if st.button("Load from Kaggle"):
            with st.spinner("Downloading from Kaggle..."):
                try:
                    nrows = kg_rows if kg_rows > 0 else None
                    df = load_kaggle_spark(kg_path, kg_file, nrows=nrows)
                    if df:
                        st.session_state.spark_df          = df
                        st.session_state.spark_df_original = df
                        st.session_state.train_df          = None
                        st.session_state.test_df           = None
                        r, c = get_shape(df)
                        log(f"Loaded Kaggle: {r:,} rows × {c} cols")
                        st.success(f"✅ Loaded {r:,} rows")
                except Exception as e:
                    st.error(f"Error: {e}")

    # Column selector
    df_current = st.session_state.spark_df
    if df_current is not None:
        st.divider()
        st.markdown("### 🔍 Column Inspector")
        all_cols     = df_current.columns
        selected_col = st.selectbox("Column", [""] + all_cols, label_visibility='collapsed')
        col_type     = get_col_type(df_current, selected_col) if selected_col else "—"
        if selected_col:
            badge_cls = 'badge-num' if col_type == 'numeric' else 'badge-cat'
            st.markdown(f'<span class="badge {badge_cls}">{col_type}</span>', unsafe_allow_html=True)

        st.divider()
        if st.button("🔄 Reset to Original"):
            if st.session_state.spark_df_original is not None:
                st.session_state.spark_df = st.session_state.spark_df_original
                st.session_state.train_df = None
                st.session_state.test_df  = None
                log("Dataset reset to original.")
                st.success("Reset!")
                st.rerun()
    else:
        selected_col = ""
        col_type     = "—"

    # Operation log
    st.divider()
    st.markdown("### 📋 Operation Log")
    log_html = "<br>".join(st.session_state.logs[-20:][::-1]) if st.session_state.logs else "No operations yet."
    st.markdown(f'<div class="log-panel">{log_html}</div>', unsafe_allow_html=True)


# ==============================================================================
# MAIN AREA
# ==============================================================================
t, pal, mode = get_theme()

if st.session_state.spark_df is None:
    hero_icon  = "🔬"
    hero_color = t["primary"]
    st.markdown(f"""
<div style="display:flex;flex-direction:column;align-items:center;justify-content:center;
     height:62vh;text-align:center;gap:18px;">
    <div style="font-size:4rem;filter:drop-shadow(0 0 24px {hero_color}66);">{hero_icon}</div>
    <div style="font-family:'Space Mono',monospace;font-size:1.9rem;font-weight:700;
                color:{hero_color};letter-spacing:-0.03em;">DataSci Studio</div>
    <div style="color:{t["muted"]};max-width:440px;line-height:1.75;font-size:0.9rem;">
        Upload a CSV or connect a Kaggle dataset from the sidebar to begin your
        end-to-end data science workflow powered by PySpark.
    </div>
    <div style="display:flex;gap:8px;margin-top:8px;flex-wrap:wrap;justify-content:center;">
        <span class="badge">PySpark Backend</span>
        <span class="badge">Scalable EDA</span>
        <span class="badge">Missing Detection</span>
        <span class="badge">Outlier Detection</span>
        <span class="badge">Encoding</span>
        <span class="badge">Feature Scaling</span>
        <span class="badge">Classification</span>
        <span class="badge">Regression</span>
        <span class="badge">Prediction</span>
    </div>
</div>
""", unsafe_allow_html=True)
else:
    df = st.session_state.spark_df
    tab_viz, tab_eda, tab_proc, tab_model = st.tabs([
        "📊  Visualization", "📈  EDA", "🧹  Processing", "🤖  Modeling",
    ])
    with tab_viz:   visualization_tab(df)
    with tab_eda:   eda_tab(df, selected_col, col_type)
    with tab_proc:  processing_tab(df)
    with tab_model: modeling_tab(df)

Overwriting app.py


### ngrok

In [120]:
from pyngrok import ngrok

ngrok.set_auth_token("3CuZ3seT0zULTvlHYx3CFo9JqPf_6cpBhDWTmYQ3mUyxiT7LP")

ngrok.kill()
public_url = ngrok.connect(8501)
print(public_url)

NgrokTunnel: "https://faceted-duplicate-reputable.ngrok-free.dev" -> "http://localhost:8501"


In [121]:
# !streamlit run app.py &>/content/logs.txt &